# 08 - Dashboard Data Generator

This notebook generates JSON data for the Next.js dashboard consumption.

**Features:**
- Key metrics summary (7/30/90 day stats)
- Sparkline data for trends
- Delay hotspots
- Recent performance highlights

**Output:** `static/data/dashboard.json`

In [1]:
import os
import sys
import json
import sqlite3
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

project_root = os.path.abspath(os.path.join(os.getcwd(), '..','..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

DB_PATH = os.path.join(project_root, 'data', 'caltrain_lat_long.db')
GTFS_PATH = os.path.join(project_root, 'gtfs_data')
METADATA_PATH = os.path.join(project_root, 'data', 'station_metadata.json')
OUTPUT_DIR = os.path.join(project_root, 'static', 'data')

print(f"Database exists: {os.path.exists(DB_PATH)}")

Database exists: True


## Load Data

In [2]:
sys.path.insert(0, os.path.join(project_root, 'src'))
from utils.time_utils import calculate_time_difference, normalize_time
from utils.geo_utils import haversine

def load_station_metadata():
    """Load station metadata."""
    if os.path.exists(METADATA_PATH):
        with open(METADATA_PATH, 'r') as f:
            data = json.load(f)
        
        stations = {}
        for s in data['stations']:
            for stop_id in s['stop_ids']:
                stations[stop_id] = s['name']
        return stations
    return {}

def load_arrivals(days=90):
    """Load recent arrivals."""
    conn = sqlite3.connect(DB_PATH)
    
    cutoff = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    
    query = f"""
    SELECT trip_id, stop_id, vehicle_lat, vehicle_lon, timestamp
    FROM train_locations
    WHERE timestamp >= '{cutoff}'
    """
    raw_df = pd.read_sql_query(query, conn)
    conn.close()
    
    if raw_df.empty:
        return pd.DataFrame()
    
    raw_df['timestamp'] = pd.to_datetime(raw_df['timestamp'], errors='coerce')
    raw_df = raw_df.dropna(subset=['timestamp'])
    print(f"Loaded {len(raw_df):,} records")
    
    # Load GTFS
    stops_df = pd.read_csv(os.path.join(GTFS_PATH, 'stops.txt'))
    stops_df = stops_df[stops_df['stop_id'].str.isnumeric()]
    stop_times_df = pd.read_csv(os.path.join(GTFS_PATH, 'stop_times.txt'))
    
    # Type conversion
    raw_df['stop_id'] = pd.to_numeric(raw_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    raw_df['trip_id'] = pd.to_numeric(raw_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    stops_df['stop_id'] = pd.to_numeric(stops_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['stop_id'] = pd.to_numeric(stop_times_df['stop_id'].astype(str), errors='coerce').astype('Int64')
    stop_times_df['trip_id'] = pd.to_numeric(stop_times_df['trip_id'].astype(str), errors='coerce').astype('Int64')
    
    raw_df = raw_df.dropna(subset=['trip_id', 'stop_id'])
    
    # Merge
    df = pd.merge(raw_df, stop_times_df[['trip_id', 'stop_id', 'arrival_time']], 
                  on=['trip_id', 'stop_id'], how='inner')
    df = pd.merge(df, stops_df[['stop_id', 'stop_name', 'stop_lat', 'stop_lon']], 
                  on='stop_id', how='inner')
    
    df['distance'] = df.apply(lambda r: haversine(
        r['vehicle_lat'], r['vehicle_lon'], r['stop_lat'], r['stop_lon']
    ), axis=1)
    
    df['date'] = df['timestamp'].dt.date
    df['arrival_time'] = df['arrival_time'].apply(normalize_time)
    
    # Arrivals
    min_dist = df.groupby(['trip_id', 'stop_id', 'date'])['distance'].min().reset_index()
    merged = pd.merge(df, min_dist, on=['trip_id', 'stop_id', 'date', 'distance'])
    arrivals = merged.groupby(['trip_id', 'stop_id', 'date']).first().reset_index()
    
    arrivals['delay_min'] = arrivals.apply(
        lambda r: calculate_time_difference(r['arrival_time'], r['timestamp']), axis=1
    )
    arrivals.loc[arrivals.delay_min > 500, 'delay_min'] = 0
    arrivals.loc[arrivals.delay_min < -100, 'delay_min'] = 0
    
    arrivals['on_time'] = arrivals['delay_min'] <= 4
    arrivals['status'] = 'On Time'
    arrivals.loc[(arrivals.delay_min > 4) & (arrivals.delay_min <= 15), 'status'] = 'Minor'
    arrivals.loc[arrivals.delay_min > 15, 'status'] = 'Major'
    
    print(f"Processed {len(arrivals):,} arrivals")
    return arrivals

station_names = load_station_metadata()
arrivals = load_arrivals(days=90)

Base directory: D:\caltrain-tracker
Dotenv path: D:\caltrain-tracker\.env
Dotenv file exists: True
Loaded 952,442 records
Processed 151,684 arrivals


## Calculate Dashboard Metrics

In [3]:
def calculate_period_stats(arrivals, days):
    """Calculate stats for a specific period."""
    cutoff = datetime.now().date() - timedelta(days=days)
    period = arrivals[arrivals['date'] >= cutoff]
    
    if period.empty:
        return {'on_time_pct': 0, 'total': 0, 'avg_delay': 0}
    
    return {
        'on_time_pct': round((period['on_time'].mean() * 100), 1),
        'total': int(len(period)),
        'avg_delay': round(period['delay_min'].mean(), 1),
    }

def calculate_trend(arrivals):
    """Calculate week-over-week trend."""
    now = datetime.now().date()
    this_week = arrivals[(arrivals['date'] >= now - timedelta(days=7)) & (arrivals['date'] < now)]
    last_week = arrivals[(arrivals['date'] >= now - timedelta(days=14)) & (arrivals['date'] < now - timedelta(days=7))]
    
    if this_week.empty or last_week.empty:
        return 'stable'
    
    this_pct = this_week['on_time'].mean() * 100
    last_pct = last_week['on_time'].mean() * 100
    diff = this_pct - last_pct
    
    if diff > 2:
        return 'improving'
    elif diff < -2:
        return 'declining'
    return 'stable'

# Calculate highlights
stats_7d = calculate_period_stats(arrivals, 7)
stats_30d = calculate_period_stats(arrivals, 30)
stats_90d = calculate_period_stats(arrivals, 90)
trend = calculate_trend(arrivals)

print(f"7-day: {stats_7d['on_time_pct']}%")
print(f"30-day: {stats_30d['on_time_pct']}%")
print(f"90-day: {stats_90d['on_time_pct']}%")
print(f"Trend: {trend}")

7-day: 92.1%
30-day: 92.5%
90-day: 91.8%
Trend: stable


## Generate Sparkline Data

In [4]:
def generate_sparkline(arrivals, days=30):
    """Generate daily on-time percentages for sparkline."""
    cutoff = datetime.now().date() - timedelta(days=days)
    period = arrivals[arrivals['date'] >= cutoff]
    
    if period.empty:
        return []
    
    daily = period.groupby('date')['on_time'].mean().reset_index()
    daily['on_time_pct'] = (daily['on_time'] * 100).round(1)
    daily = daily.sort_values('date')
    
    return [
        {'date': str(row['date']), 'value': row['on_time_pct']}
        for _, row in daily.iterrows()
    ]

sparkline = generate_sparkline(arrivals, 30)
print(f"Sparkline: {len(sparkline)} data points")
sparkline[:5]

Sparkline: 29 data points


[{'date': '2025-11-30', 'value': 97.0},
 {'date': '2025-12-01', 'value': 94.9},
 {'date': '2025-12-02', 'value': 92.4},
 {'date': '2025-12-03', 'value': 96.6},
 {'date': '2025-12-04', 'value': 98.4}]

## Find Delay Hotspots

In [5]:
def find_hotspots(arrivals, days=7, top_n=5):
    """Find stations with worst performance recently."""
    cutoff = datetime.now().date() - timedelta(days=days)
    period = arrivals[arrivals['date'] >= cutoff]
    
    if period.empty:
        return []
    
    # Map to parent station names
    period = period.copy()
    period['station'] = period['stop_id'].apply(
        lambda x: station_names.get(str(x), period.loc[period['stop_id'] == x, 'stop_name'].iloc[0] if len(period[period['stop_id'] == x]) > 0 else 'Unknown')
    )
    
    # Group by station
    stats = period.groupby('station').agg(
        on_time_pct=('on_time', lambda x: round(x.mean() * 100, 1)),
        total=('on_time', 'count'),
        avg_delay=('delay_min', lambda x: round(x.mean(), 1)),
    ).reset_index()
    
    # Filter and sort
    stats = stats[stats['total'] >= 10]  # Minimum arrivals
    worst = stats.nsmallest(top_n, 'on_time_pct')
    
    return [
        {
            'station': row['station'],
            'on_time_pct': row['on_time_pct'],
            'arrivals': int(row['total']),
            'avg_delay': row['avg_delay'],
        }
        for _, row in worst.iterrows()
    ]

hotspots = find_hotspots(arrivals, 7, 5)
print("Delay Hotspots (last 7 days):")
for h in hotspots:
    print(f"  {h['station']}: {h['on_time_pct']}% on-time")

Delay Hotspots (last 7 days):
  Tamien: 68.8% on-time
  Blossom Hill: 81.2% on-time
  Capitol: 81.2% on-time
  College Park: 83.3% on-time
  Bayshore: 86.6% on-time


## Generate Dashboard JSON

In [6]:
def generate_dashboard_json():
    """Generate complete dashboard JSON."""
    
    dashboard = {
        'last_updated': datetime.now().isoformat(),
        'highlights': {
            'on_time_pct_7d': stats_7d['on_time_pct'],
            'on_time_pct_30d': stats_30d['on_time_pct'],
            'on_time_pct_90d': stats_90d['on_time_pct'],
            'total_arrivals_30d': stats_30d['total'],
            'avg_delay_30d': stats_30d['avg_delay'],
            'trend': trend,
        },
        'sparkline': sparkline,
        'hotspots': hotspots,
        'status_breakdown': {
            'on_time': int((arrivals['status'] == 'On Time').sum()),
            'minor': int((arrivals['status'] == 'Minor').sum()),
            'major': int((arrivals['status'] == 'Major').sum()),
        }
    }
    
    return dashboard

dashboard_data = generate_dashboard_json()
print(json.dumps(dashboard_data, indent=2)[:1000] + '...')

{
  "last_updated": "2025-12-30T18:13:59.635143",
  "highlights": {
    "on_time_pct_7d": 92.1,
    "on_time_pct_30d": 92.5,
    "on_time_pct_90d": 91.8,
    "total_arrivals_30d": 48518,
    "avg_delay_30d": -0.9,
    "trend": "stable"
  },
  "sparkline": [
    {
      "date": "2025-11-30",
      "value": 97.0
    },
    {
      "date": "2025-12-01",
      "value": 94.9
    },
    {
      "date": "2025-12-02",
      "value": 92.4
    },
    {
      "date": "2025-12-03",
      "value": 96.6
    },
    {
      "date": "2025-12-04",
      "value": 98.4
    },
    {
      "date": "2025-12-05",
      "value": 95.1
    },
    {
      "date": "2025-12-06",
      "value": 88.3
    },
    {
      "date": "2025-12-07",
      "value": 82.1
    },
    {
      "date": "2025-12-08",
      "value": 94.3
    },
    {
      "date": "2025-12-09",
      "value": 90.6
    },
    {
      "date": "2025-12-10",
      "value": 90.4
    },
    {
      "date": "2025-12-11",
      "value": 92.0
    },
    {
    

## Export JSON

In [7]:
os.makedirs(OUTPUT_DIR, exist_ok=True)
output_path = os.path.join(OUTPUT_DIR, 'dashboard.json')

with open(output_path, 'w') as f:
    json.dump(dashboard_data, f, indent=2)

print(f"✅ Exported to: {output_path}")
print(f"   File size: {os.path.getsize(output_path) / 1024:.1f} KB")

✅ Exported to: d:\caltrain-tracker\static\data\dashboard.json
   File size: 2.8 KB


## Preview Dashboard Data

In [8]:
# Pretty print for review
print("=" * 60)
print("📊 DASHBOARD DATA PREVIEW")
print("=" * 60)

print("\n🎯 Highlights:")
h = dashboard_data['highlights']
print(f"  7-day on-time:  {h['on_time_pct_7d']}%")
print(f"  30-day on-time: {h['on_time_pct_30d']}%")
print(f"  90-day on-time: {h['on_time_pct_90d']}%")
print(f"  Total arrivals: {h['total_arrivals_30d']:,}")
print(f"  Trend: {h['trend']}")

print("\n📈 Sparkline (last 5 days):")
for point in dashboard_data['sparkline'][-5:]:
    print(f"  {point['date']}: {point['value']}%")

print("\n🔥 Hotspots:")
for spot in dashboard_data['hotspots']:
    print(f"  {spot['station']}: {spot['on_time_pct']}%")

print("\n📊 Status Breakdown:")
s = dashboard_data['status_breakdown']
total = s['on_time'] + s['minor'] + s['major']
print(f"  On Time: {s['on_time']:,} ({s['on_time']/total*100:.1f}%)")
print(f"  Minor:   {s['minor']:,} ({s['minor']/total*100:.1f}%)")
print(f"  Major:   {s['major']:,} ({s['major']/total*100:.1f}%)")

📊 DASHBOARD DATA PREVIEW

🎯 Highlights:
  7-day on-time:  92.1%
  30-day on-time: 92.5%
  90-day on-time: 91.8%
  Total arrivals: 48,518
  Trend: stable

📈 Sparkline (last 5 days):
  2025-12-24: 89.8%
  2025-12-25: 85.7%
  2025-12-26: 92.8%
  2025-12-27: 97.6%
  2025-12-28: 84.6%

🔥 Hotspots:
  Tamien: 68.8%
  Blossom Hill: 81.2%
  Capitol: 81.2%
  College Park: 83.3%
  Bayshore: 86.6%

📊 Status Breakdown:
  On Time: 139,268 (91.8%)
  Minor:   10,443 (6.9%)
  Major:   1,973 (1.3%)
